# Rúbrica ponderada

Pesos fijados de antemano, iguales a los del proyecto anterior del curso: MSE en validación 25 %, MSE en prueba 15 %, CSI de picos en validación 15 %, CSI en prueba 15 %, AUC de picos en validación 10 %, AUC en prueba 10 %, significancia frente a los tres modelos tradicionales 10 %. Cada criterio se escala contra el mejor modelo (100 = el mejor). Niveles: Excelente ≥ 90, Bueno 75–90, Aceptable 60–75, Insuficiente < 60.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, "../src"); import estilo_tablas; estilo_tablas.activar(); import pipeline_energia as P
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
R = "../resultados/"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})
COL = {"Persistencia (misma hora de ayer)": "#9AA5A0", "Estacional (misma hora semana pasada)": "#B8C2BD", "Promedio por hora y día": "#D5DBD8",
       "Ridge (lineal)": "#5B7F96", "Random forest (relativo)": "#1F5A3A", "XGBoost (relativo)": "#C8892B", "Híbrido Ridge + XGBoost (residuo relativo)": "#8E4B7A"}

In [2]:
val = pd.read_csv(R + "validacion.csv").set_index("modelo"); tp = pd.read_csv(R + "prueba.csv").set_index("modelo"); dm = pd.read_csv(R + "diebold_mariano.csv")
TRAD = P.TRADICIONALES; M = list(val.index)
sig = {m: np.mean([((dm.modelo == m) & (dm.referencia == t) & (dm.DM > 0) & (dm.p < 0.05)).any() for t in TRAD if t != m]) for m in M}
def esc(v, menor=True): v = pd.Series(v); return (100*v.min()/v) if menor else (100*v/v.max())
def rub(W):
    c = pd.DataFrame(index=M)
    c["MSE_val"] = esc(val.loc[M, "MSE"]); c["MSE_test"] = esc(tp.loc[M, "MSE"])
    c["CSI_val"] = esc(val.loc[M, "CSI"], False); c["CSI_test"] = esc(tp.loc[M, "CSI"], False)
    c["AUC_val"] = esc(val.loc[M, "AUC_pico"] - .5, False); c["AUC_test"] = esc(tp.loc[M, "AUC_pico"] - .5, False)
    c["signif"] = pd.Series(sig)*100
    c["puntaje"] = sum(c[k]*w for k, w in W.items())
    c["nivel"] = pd.cut(c.puntaje, [-1, 60, 75, 90, 101], labels=["Insuficiente", "Aceptable", "Bueno", "Excelente"])
    return c.sort_values("puntaje", ascending=False)
W = {"MSE_val": .25, "MSE_test": .15, "CSI_val": .15, "CSI_test": .15, "AUC_val": .10, "AUC_test": .10, "signif": .10}
R0 = rub(W); R0.to_csv(R + "rubrica.csv"); R0.round(1)

,MSE_val,MSE_test,CSI_val,CSI_test,AUC_val,AUC_test,signif,puntaje,nivel
Random forest (relativo),"100,0","100,0","91,4","100,0","98,9","100,0","100,0","98,60",Excelente
XGBoost (relativo),"98,4","97,1","90,6","100,0","97,2","99,8","100,0","97,50",Excelente
Híbrido Ridge + XGBoost (residuo relativo),"95,6","76,3","73,8","93,1","100,0","99,4","100,0","90,30",Excelente
Ridge (lineal),"86,5","65,8","69,7","89,9","98,8","98,3","100,0","85,10",Bueno
Persistencia (misma hora de ayer),"38,3","36,0","100,0","86,6","92,5","89,9","100,0","71,20",Aceptable
Estacional (misma hora semana pasada),"33,0","27,8","83,9","82,3","85,0","85,6","50,0","59,40",Insuficiente
Promedio por hora y día,"19,4","4,4","0,0","0,0","87,1","69,8","0,0","21,20",Insuficiente


In [3]:
alt = {"pesos iguales": {k: 1/7 for k in W}, "solo error": {"MSE_val": .5, "MSE_test": .5, "CSI_val": 0, "CSI_test": 0, "AUC_val": 0, "AUC_test": 0, "signif": 0},
       "solo picos": {"MSE_val": 0, "MSE_test": 0, "CSI_val": .25, "CSI_test": .25, "AUC_val": .25, "AUC_test": .25, "signif": 0}}
pd.DataFrame({k: rub(w).puntaje.rank(ascending=False).astype(int) for k, w in alt.items()} | {"rúbrica": R0.puntaje.rank(ascending=False).astype(int)})

,pesos iguales,solo error,solo picos,rúbrica
Estacional (misma hora semana pasada),6,6,6,6
Híbrido Ridge + XGBoost (residuo relativo),3,3,4,3
Persistencia (misma hora de ayer),5,5,3,5
Promedio por hora y día,7,7,7,7
Random forest (relativo),1,1,1,1
Ridge (lineal),4,4,5,4
XGBoost (relativo),2,2,2,2


 La rúbrica coloca primero al **random forest relativo** (98,6, Excelente), seguido del XGBoost relativo (97,5) y del híbrido (90,3); Ridge queda en Bueno (85,1), la persistencia en Aceptable (71,2) y los otros dos tradicionales en Insuficiente. El random forest relativo es primero con todos los esquemas de pesos. Solo cuando se califican únicamente los picos la persistencia sube al tercer lugar, porque detecta muchas horas altas aunque con el doble de falsas alarmas.